# Minimal RAG Pipeline

This notebook loads documents, chunks them, stores embeddings in Pinecone, retrieves similar chunks for a query, and generates an answer with an LLM.


## Imports

This cell loads the shared helpers from `rag_pipeline.py` and the small utilities needed in the notebook.


In [1]:
from pathlib import Path

from rag_pipeline import (
    DEFAULT_CHAT_MODEL,
    DEFAULT_EMBEDDING_MODEL,
    PINECONE_INDEX_NAME,
    answer_query,
    build_chunks,
    embed_texts,
    index_documents,
    retrieve_context,
)


## Environment and Config

This cell defines the document directory, namespace, and query used throughout the notebook.


In [ ]:
documents_dir = Path("./docs")
namespace = "default"

print(f"Chat model: {DEFAULT_CHAT_MODEL}")
print(f"Embedding model: {DEFAULT_EMBEDDING_MODEL}")
print(f"Pinecone index: {PINECONE_INDEX_NAME}")
print(f"Documents directory: {documents_dir.resolve()}")


Chat model: llama-3.3-70b-versatile
Embedding model: all-MiniLM-L6-v2
Pinecone index: rag-minimal
Documents directory: D:\GitREPOS\RAG\docs


## Document Loading and Chunking

This section reads text from each supported file type and splits it into overlapping chunks for embedding.


In [3]:
chunks = build_chunks(documents_dir)
print(f"Loaded {len(chunks)} chunks.")
if chunks:
    print("First chunk source:", chunks[0].source)
    print("First chunk preview:", chunks[0].text[:300])


Loaded 101 chunks.
First chunk source: RAG+KG.pdf
First chunk preview: Empowering LLMs by hybrid retrieval-augmented generation for domain-centric Q & A in smart manufacturing ☆ Yuwei Wan a , Zheyuan Chen b , c , Ying Liu a , * , Chong Chen d , Michael Packianather a a Department of Mechanical Engineering, School of Engineering, Cardiff University, Cardiff CF24 3AA, UK


## Pinecone and Embeddings

This section creates the embedding model and converts text into vectors.


In [4]:
if chunks:
    sample_embedding = embed_texts([chunks[0].text])[0]
    print(f"Sample embedding dimension: {len(sample_embedding)}")
else:
    print("No supported documents were found in ./docs.")


d:\Miniconda\envs\rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4319.10it/s]


Sample embedding dimension: 384


## Indexing and Retrieval

This section stores chunks in Pinecone and retrieves the nearest matches for a query.


In [5]:
indexed = index_documents(documents_dir, namespace=namespace)
print(f"Indexed {indexed} chunks.")

contexts = retrieve_context(query, top_k=5, namespace=namespace)
for item in contexts:
    print(f"[{item['score']:.4f}] {item['source']}#{item['chunk_index']}")
    print(item["text"])
    print()


Indexed 101 chunks.
[0.4099] RAG+KG.pdf#54
ber of edges 5815 Table 5 The descriptions of typical semantic relations regarding DfAM. Relations Symbols Descriptions HasMaterialProperty r 1 Links materials to their specific properties or attributes. SuitableForProcess r 2 Associates materials or designs with compatible AM processes. HasGeometryFeature r 3 Links design to their geometric features or complexities. RequiresSupport r 4 Indicates that a design requires support structures during printing. EnhancesPerformance r 5 Denotes design modifications that improve performance metrics. CompatibleWithMachine r 6 Associates materials or designs with specific AM equipment. Y. Wan et al. Advanced Engineering Informatics 65 (2025) 103212 9 performances of the proposed KG-Vector approach in comparison with different RAG approaches and diverse LLMs, including generic RAG metrics and domain metrics. (1) Generic RAG metrics: the evaluation of the proposed hybrid KG- Vector RAG approach is conducted

## Example Usage

This final cell shows the end-to-end question answering step using the retrieved context.


In [7]:
query = "What does the paper say about KG?"

answer = answer_query(query, top_k=5, namespace=namespace, contexts=contexts)
print(answer)


The paper discusses KG (Knowledge Graph) in the context of a hybrid RAG (Retrieval-Augmented Generation) approach, specifically the KG-Vector RAG approach (Source: RAG+KG.pdf, Chunk: 54). It is used to enhance the performance of Language Models (LLMs) by incorporating structured knowledge from a domain-specific KG during the retrieval process (Source: RAG+KG.pdf, Chunk: 47). The KG is used to retrieve and aggregate information to provide additional context for the LLM, improving the accuracy and relevance of the responses (Source: RAG+KG.pdf, Chunk: 47). The effectiveness of the proposed KG-Vector RAG approach is demonstrated by comparing different RAG methods within the same LLM (Source: RAG+KG.pdf, Chunk: 74).
